In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.cluster import KMeans
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
import xgboost as xgb

In [3]:
np.random.seed(42)
n_students = 240
student_ids = [f"S{1000+i}" for i in range(n_students)]
students = pd.DataFrame({
"student_id": student_ids,
"gender": np.random.choice(["Female", "Male", "F", "M", "female", "male", None], n_students,
p=[0.28, 0.28, 0.10, 0.10, 0.10, 0.10, 0.04]),
"department": np.random.choice(["AI Engineering", "Computer Engineering", "Software Engineering",
"Industrial Engineering", "AI Eng.", "Comp Eng", None], n_students,
p=[0.30, 0.25, 0.18, 0.15, 0.06, 0.04, 0.02]),
"year": np.random.choice([1, 2, 3, 4], n_students, p=[0.45, 0.35, 0.12, 0.08]),
"scholarship_rate": np.random.choice([0, 25, 50, 75, 100, np.nan], n_students,
p=[0.30, 0.20, 0.20, 0.12, 0.15, 0.03])

})
weeks = pd.date_range("2025-02-03", periods=12, freq="W-MON")
activity_rows = []
for sid in student_ids:
    base_engagement = np.random.normal(6.5, 1.8)
    for week in weeks:
        study_hours = max(0, np.random.normal(base_engagement, 2.0))
        lms_logins = max(0, int(np.random.normal(base_engagement * 2.5, 5)))
        attendance_rate = np.clip(np.random.normal(0.78, 0.18), 0, 1)
        activity_rows.append([sid, week, study_hours, lms_logins, attendance_rate])
        activity = pd.DataFrame(activity_rows, columns=[
        "student_id", "week_start", "study_hours", "lms_logins", "attendance_rate"
])
# Intentional missing values
activity.loc[np.random.choice(activity.index, 50, replace=False), "study_hours"] = np.nan
activity.loc[np.random.choice(activity.index, 40, replace=False), "attendance_rate"] = np.nan
# Intentional impossible values and outliers
activity.loc[np.random.choice(activity.index, 8, replace=False), "study_hours"] = 80
activity.loc[np.random.choice(activity.index, 6, replace=False), "attendance_rate"] = 1.35
activity.loc[np.random.choice(activity.index, 6, replace=False), "lms_logins"] = -5
assessment = pd.DataFrame({
"student_id": student_ids,
"midterm_score": np.random.normal(68, 15, n_students),
"project_score": np.random.normal(72, 14, n_students),
"quiz_average": np.random.normal(70, 13, n_students)
})
# Intentional impossible assessment scores
assessment.loc[np.random.choice(assessment.index, 5, replace=False), "midterm_score"] = 130
assessment.loc[np.random.choice(assessment.index, 5, replace=False), "project_score"] = -10
assessment.loc[np.random.choice(assessment.index, 10, replace=False), "quiz_average"] = np.nan
campus = pd.DataFrame({
"student_id": student_ids,

"club_participation": np.random.choice(["Yes", "No", "yes", "no", "Y", "N", None], n_students,
p=[0.22, 0.38, 0.08, 0.12, 0.07, 0.10, 0.03]),
"library_visits": np.random.poisson(7, n_students),
"advisor_meetings": np.random.poisson(2, n_students)
})
# Generate final score with signal from engagement and assessments
activity_summary = activity.groupby("student_id").agg(
avg_study_hours=("study_hours", "mean"),
avg_lms_logins=("lms_logins", "mean"),
avg_attendance_rate=("attendance_rate", "mean")
).reset_index()
final = students.merge(activity_summary, on="student_id", how="left").merge(assessment, on="student_id",
how="left")
noise = np.random.normal(0, 7, n_students)
final_score = (
0.30 * final["midterm_score"].fillna(final["midterm_score"].median()) +
0.25 * final["project_score"].fillna(final["project_score"].median()) +
0.20 * final["quiz_average"].fillna(final["quiz_average"].median()) +
3.0 * final["avg_study_hours"].fillna(final["avg_study_hours"].median()) +
10.0 * final["avg_attendance_rate"].fillna(final["avg_attendance_rate"].median()) +
noise
)
grades = pd.DataFrame({
"student_id": student_ids,
"final_score": np.clip(final_score, 0, 100)
})
grades["risk_status"] = np.where(grades["final_score"] < 60, "At_Risk", "Not_At_Risk")
# Save raw files
students.to_csv("hw4_students_raw.csv", index=False)
activity.to_csv("hw4_weekly_activity_raw.csv", index=False)
assessment.to_csv("hw4_assessment_raw.csv", index=False)
campus.to_csv("hw4_campus_raw.csv", index=False)
grades.to_csv("hw4_grades_raw.csv", index=False)
print("Synthetic raw datasets created successfully.")

Synthetic raw datasets created successfully.


In [7]:
student_df = pd.read_csv("hw4_students_raw.csv")
w_activity_df = pd.read_csv("hw4_weekly_activity_raw.csv")
assessment_df = pd.read_csv("hw4_assessment_raw.csv")
campus_df = pd.read_csv("hw4_campus_raw.csv")
grades_df = pd.read_csv("hw4_grades_raw.csv

# Assigning every csv documents to pandas DataFrame.

In [40]:
datas = [student_df, w_activity_df, assessment_df, campus_df, grades_df]
names = ["students.csv", "w_activity.csv", "assessment.csv", "campus.csv", "grades.csv"]

for name, data in zip(names, datas):
    print(f" The shape of {name}: {data.shape}")
#Checking shapes of every datasets.

 The shape of students.csv: (240, 5)
 The shape of w_activity.csv: (2880, 5)
 The shape of assessment.csv: (240, 4)
 The shape of campus.csv: (240, 4)
 The shape of grades.csv: (240, 3)


In [41]:
for name, data in zip(names, datas):
    print(name)
    print(data.head(5))
    print("-"*75)

students.csv
  student_id  gender            department  year  scholarship_rate
0      S1000    Male              Comp Eng     2             100.0
1      S1001    male              Comp Eng     2               0.0
2      S1002       M               AI Eng.     2               0.0
3      S1003       F  Computer Engineering     2              75.0
4      S1004  Female        AI Engineering     2              50.0
---------------------------------------------------------------------------
w_activity.csv
  student_id  week_start  study_hours  lms_logins  attendance_rate
0      S1000  2025-02-03     5.325414          14         0.526801
1      S1000  2025-02-10     4.989714          10         0.604343
2      S1000  2025-02-17     8.943464          12         1.000000
3      S1000  2025-02-24     7.822816          18         0.625496
4      S1000  2025-03-03     8.236800          14         0.801962
---------------------------------------------------------------------------
assessment.csv
 

In [42]:
for name, data in zip(names, datas):
    print("|" + name + "| Columns |")
    print(data.columns)
    print("-"*90)

|students.csv| Columns |
Index(['student_id', 'gender', 'department', 'year', 'scholarship_rate'], dtype='object')
------------------------------------------------------------------------------------------
|w_activity.csv| Columns |
Index(['student_id', 'week_start', 'study_hours', 'lms_logins',
       'attendance_rate'],
      dtype='object')
------------------------------------------------------------------------------------------
|assessment.csv| Columns |
Index(['student_id', 'midterm_score', 'project_score', 'quiz_average'], dtype='object')
------------------------------------------------------------------------------------------
|campus.csv| Columns |
Index(['student_id', 'club_participation', 'library_visits',
       'advisor_meetings'],
      dtype='object')
------------------------------------------------------------------------------------------
|grades.csv| Columns |
Index(['student_id', 'final_score', 'risk_status'], dtype='object')
-----------------------------------------

In [43]:
for name, data in zip(names, datas):
    print("|" + name + "| Columns |")
    print(data.dtypes)
    print("-"*90)

|students.csv| Columns |
student_id           object
gender               object
department           object
year                  int64
scholarship_rate    float64
dtype: object
------------------------------------------------------------------------------------------
|w_activity.csv| Columns |
student_id          object
week_start          object
study_hours        float64
lms_logins           int64
attendance_rate    float64
dtype: object
------------------------------------------------------------------------------------------
|assessment.csv| Columns |
student_id        object
midterm_score    float64
project_score    float64
quiz_average     float64
dtype: object
------------------------------------------------------------------------------------------
|campus.csv| Columns |
student_id            object
club_participation    object
library_visits         int64
advisor_meetings       int64
dtype: object
------------------------------------------------------------------------------

#TASK1.2
(1)In the student.csv for each unique object represents students and there are 4 (gender,department,year,scholarship_rate) attiributes.

(2) In the w_activity.csv for each unique object(students) again there are 4 (week_start,study_hours,lms_logins,attendance_rate) attributes.

(3) In the assessment.csv for each unique object(students) there are 3
(midterm_score,project_score,quiz_average) attributes.

(4) In the campus.csv for each object(students) there are 3 (club_participation  library_visits,advisor_meetings) attributes.

(5) In the grades.csv for each object(students) there are 2(final_score,  risk_status) attributes.

-------------------------------------------------------------------------------------
1. student_id -> Nominal dtype -> discrete
2. risk_status -> Nominal dtype -> discrete
3. attendance_rate -> Ratio numberic -> continuous
4. week_start -> Date/Time -> discrete
5. lms_logins -> Ratio numeric count -> discrete
6. gender -> Nominal dtype -> discrete
7. library_visits -> Ratio numeric count -> discrete
8. study_hours -> Ratio numeric -> continuous
9. club_participation -> Nominal categorical -> discrete
10. quiz_average -> Ratio numeric -> continuous
-------------------------------------------------------------------------------------